# 02 — MLP preliminar (MLP-A-128)

Experimento 1 (fase de acompanhamento): conversão das imagens RGB 128 × 128 em vetor, primeira MLP, treino com monitoramento de **F1 macro na validação**, checkpoint, histórico e curvas.

> O **teste não é usado** neste notebook (regra do acompanhamento). Esta execução é identificada como `preliminar`, separada das seis execuções finais.

Pré-requisitos: `dados/raw/NWPU-RESISC45/` filtrado (`src/setup_dataset.py`) e `dados/splits/particoes.csv` já gerado.

## 0. (Opcional) Google Colab

Descomente para clonar o repositório e apontar para o dataset no Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !git clone https://github.com/<org>/<repo>.git /content/projeto
# %cd /content/projeto
# !mkdir -p dados/raw && cp -r /content/drive/MyDrive/NWPU-RESISC45 dados/raw/

## 1. Imports e semente

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from dados_imagens import CLASSES, carregar_particoes, preparar_dados_mlp
from modelos_mlp import MLP_CONFIGS, criar_mlp, resumo
from treino import ConfigTreino, fixar_semente, treinar, obter_dispositivo, descrever_hardware
from graficos import plotar_curvas

SEMENTE = 42
fixar_semente(SEMENTE)
print('torch', torch.__version__, '|', descrever_hardware(obter_dispositivo()))

## 2. Imagem → vetor

Cada imagem é lida do **original**, convertida para RGB, redimensionada (bilinear) para 128 × 128, dividida por 255 e padronizada por canal com média/desvio **calculados só no treino**. Depois o tensor `(3, 128, 128)` é achatado em um vetor de **49.152** valores.

In [ ]:
dados, norm = preparar_dados_mlp(resolucao=128)   # carrega treino e validação (não o teste)
X_tr, y_tr, cam_tr = dados['treino']
X_va, y_va, cam_va = dados['validacao']

print('Treino   :', X_tr.shape, X_tr.dtype)
print('Validação:', X_va.shape)
print('Média por canal (treino):', np.round(norm.media, 4))
print('Desvio por canal (treino):', np.round(norm.desvio, 4))
print(pd.Series(y_tr).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES))

Verificação: desfazer o achatamento de um vetor deve reproduzir a imagem.

In [ ]:
i = 0
img = X_tr[i].reshape(3, 128, 128).transpose(1, 2, 0) * norm.desvio + norm.media
plt.figure(figsize=(3, 3)); plt.imshow(np.clip(img, 0, 1)); plt.axis('off')
plt.title(f'{CLASSES[y_tr[i]]} (vetor {X_tr.shape[1]:,} → 128×128×3)'); plt.show()

## 3. Arquitetura MLP-A

`49.152 → 512 → 128 → 10`, cada camada oculta com Linear → BatchNorm → ReLU → Dropout(0,5). A MLP-B (mais profunda: 512-256-128-64) já está em `MLP_CONFIGS` para comparação.

In [ ]:
modelo = criar_mlp('A', resolucao=128, n_classes=len(CLASSES))
print(resumo(modelo, X_tr.shape[1]))

## 4. Treino

| Item | Valor |
|---|---|
| Perda | CrossEntropyLoss |
| Otimizador | Adam, lr = 1e-4, weight decay = 1e-4 |
| Lote | 64 |
| Limite de épocas | 60 |
| Parada | 10 épocas sem melhora no **F1 macro de validação** |
| Checkpoint | época de maior F1 macro de validação |
| Aumento de dados | nenhum |
| Semente | 42 |

In [ ]:
cfg = ConfigTreino(
    nome_execucao='MLP-A-128_preliminar',
    taxa_aprendizagem=1e-4, weight_decay=1e-4, tamanho_lote=64,
    max_epocas=60, paciencia=10, semente=SEMENTE, tipo_execucao='preliminar',
    extras={'arquitetura': 'MLP-A-128', **MLP_CONFIGS['A']},
)
saida = treinar(modelo, (X_tr, y_tr), (X_va, y_va), cfg,
                dir_modelos=RAIZ / 'modelos', dir_resultados=RAIZ / 'resultados')
pd.Series(saida['resumo'])

## 5. Curvas de aprendizagem

In [ ]:
fig_path = plotar_curvas(saida['historico'], cfg.nome_execucao,
                         RAIZ / 'resultados' / 'figuras' / f'{cfg.nome_execucao}_curvas.png')
from IPython.display import Image as IPImage
IPImage(filename=str(fig_path))

In [ ]:
saida['historico'].round(4)

## 6. Leitura dos resultados (preencher após rodar)

- F1 macro / acurácia de validação no checkpoint: …
- Distância entre curvas de treino e validação (sobreajuste?): …
- Custo: parâmetros, tamanho do `.pt`, tempo médio por época: …
- Próximos passos: MLP-B, ajuste de lr/dropout (buscas identificadas como preliminares), treino final no Colab.